# 🏆 Simulación Predictiva del Mundial 2026

**Modelo basado en XGBoost Regressor (Poisson) + Monte Carlo (1000 simulaciones)**

### Descripción del Proyecto
Este notebook realiza una **simulación predictiva completa** del Mundial 2026, enfocándose en las fases eliminatorias (Octavos y Cuartos de Final).

### ¿Qué se utilizó?
- **Datos**: Partidos históricos (`datos_historicos.csv`), Ranking FIFA actualizado y Valor de Mercado de jugadores (`transfermarkt.csv`).
- **Modelo principal**: **XGBoost Regressor** con distribución **Poisson** (ideal para predecir cantidad de goles).
- **Features clave**: Diferencia de ranking, valor de mercado, forma reciente (últimos 5-10 partidos), racha y variables derivadas.
- **Simulación**: **Monte Carlo con 1000 repeticiones** para calcular probabilidades reales de avance a Cuartos de Final.
- **Métricas del modelo**: MAE ~1.0 goles, Precisión 1X2 ~60%.

### Objetivo
Predecir de forma probabilística qué equipos tienen más chances de avanzar desde Octavos hasta Cuartos, basado en datos históricos y estadísticas actuales.


In [ ]:
import pandas as pd
import numpy as np

# Cargar
df_h = pd.read_csv('/content/drive/MyDrive/Simulaciones_Mundial/Data/datos_historicos.csv',
                   encoding='utf-8', low_memory=False)

print("Dimensiones originales:", df_h.shape)

# ====================== LIMPIEZA ======================
# 1. Convertir Fecha a datetime
df_h['Fecha'] = pd.to_datetime(df_h['Fecha'], errors='coerce')

# 2. Normalizar nombres de equipos (importante para cruces)
def normalizar_equipo(x):
    if pd.isna(x): return x
    x = str(x).strip()
    # Correcciones comunes
    replacements = {
        'EEUU': 'Estados Unidos',
        'Chequia': 'República Checa',
        'Corea del Sur': 'República de Corea',
        'Arabia Saudita': 'Arabia Saudí',
        'Irán': 'RI de Irán'
    }
    for old, new in replacements.items():
        if old in x:
            x = x.replace(old, new)
    return x

for col in ['Equipo_Local', 'Equipo_Visitante']:
    df_h[col] = df_h[col].apply(normalizar_equipo)

# 3. Convertir columnas numéricas
numeric_cols = df_h.select_dtypes(include=['object']).columns
for col in numeric_cols:
    if col not in ['Fecha', 'Equipo_Local', 'Equipo_Visitante', 'Resultado_1X2']:
        df_h[col] = pd.to_numeric(df_h[col], errors='coerce')

# 4. Crear resultado numérico (opcional pero muy útil)
def resultado_num(r):
    if r == '1': return 1
    elif r == 'X': return 0
    elif r == '2': return -1
    else: return np.nan

df_h['Resultado_Num'] = df_h['Resultado_1X2'].apply(resultado_num)

# 5. Eliminar filas sin fecha o sin equipos
df_h = df_h.dropna(subset=['Fecha', 'Equipo_Local', 'Equipo_Visitante'])

print("Dimensiones después de limpieza:", df_h.shape)
print("\nColumnas disponibles:")
print(df_h.columns.tolist())

# Guardar versión limpia
df_h.to_csv('/content/drive/MyDrive/Simulaciones_Mundial/Data/datos_historicos_limpio.csv', index=False)
print("✅ Archivo guardado: datos_historicos_limpio.csv")

Dimensiones originales: (1396, 48)
Dimensiones después de limpieza: (1396, 49)

Columnas disponibles:
['Fecha', 'Equipo_Local', 'Equipo_Visitante', 'Goles_Local', 'Goles_Visitante', 'Peso_Local', 'avg_Goles_esperados_(xG)_total_Local', 'avg_Tarjetas_amarillas_total_Local', 'avg_Faltas_total_Local', 'avg_Remates_a_puerta_3_Local', 'avg_Córneres_3_Local', 'avg_Tarjetas_amarillas_3_Local', 'avg_Faltas_3_Local', 'avg_Paradas_3_Local', 'avg_Pases_Pct_3_Local', 'avg_Pases_Exitosos_3_Local', 'Peso_Visitante', 'Resultado_1X2', 'avg_Goles_esperados_(xG)_total_Visitante', 'avg_Tarjetas_amarillas_total_Visitante', 'avg_Faltas_total_Visitante', 'avg_Pases_Pct_total_Visitante', 'avg_xG_Share_total_Visitante', 'avg_Goles_esperados_(xG)_3_Visitante', 'avg_Remates_a_puerta_3_Visitante', 'avg_Córneres_3_Visitante', 'avg_Tarjetas_amarillas_3_Visitante', 'avg_Faltas_3_Visitante', 'avg_Paradas_3_Visitante', 'avg_Pases_Exitosos_3_Visitante', 'avg_Goles_3_Visitante', 'avg_Ptos_3_Visitante', 'avg_xG_Share_3_

In [ ]:
import pandas as pd
from datetime import datetime

df_r = pd.read_csv('/content/drive/MyDrive/Simulaciones_Mundial/Data/ranking_fifa.csv',
                   encoding='utf-8')

print("Original:", df_r.shape)

# Limpiar
df_r['Fecha'] = pd.to_datetime(df_r['Fecha'], format='%d-%m-%Y', errors='coerce')
df_r['País'] = df_r['País'].str.strip()

# Normalizar algunos nombres
df_r['País'] = df_r['País'].replace({
    'EEUU': 'Estados Unidos',
    'Chequia': 'República Checa',
    'Corea del Sur': 'República de Corea',
    'Arabia Saudita': 'Arabia Saudí'
})

# Ordenar por fecha descendente
df_r = df_r.sort_values(['País', 'Fecha'], ascending=[True, False])

# Quedarnos con el ranking más reciente por país
df_r_latest = df_r.drop_duplicates(subset='País', keep='first')

print("Ranking más reciente por país:", df_r_latest.shape)
print(df_r_latest.head())

df_r_latest.to_csv('/content/drive/MyDrive/Simulaciones_Mundial/Data/ranking_fifa_limpio.csv', index=False)
print("✅ Archivo guardado: ranking_fifa_limpio.csv")

Original: (6931, 3)
Ranking más reciente por país: (217, 3)
                  País  Puntuación      Fecha
6787  ARY de Macedonia     1343.00 2018-12-20
168         Afganistán      972.75 2026-04-01
63             Albania     1388.06 2026-04-01
9             Alemania     1730.37 2026-04-01
172            Andorra      945.34 2026-04-01
✅ Archivo guardado: ranking_fifa_limpio.csv


In [ ]:
import pandas as pd

df_t = pd.read_csv('/content/drive/MyDrive/Simulaciones_Mundial/Data/transfermarkt.csv',
                   encoding='utf-8')

print("Original:", df_t.shape)

# Limpieza básica
df_t['Equipo'] = df_t['Equipo'].str.strip()

# Normalizar nombres
df_t['Equipo'] = df_t['Equipo'].replace({
    'Estados Unidos': 'EEUU',
    'Chequia': 'República Checa',
    'Arabia Saudita': 'Arabia Saudí',
    'Corea del Sur': 'República de Corea'
})

df_t = df_t.sort_values('Valor_Mercado_Millones_Eur', ascending=False)

print("Top 10 valores de mercado:")
print(df_t.head(10)[['Equipo', 'Valor_Mercado_Millones_Eur']])

df_t.to_csv('/content/drive/MyDrive/Simulaciones_Mundial/Data/transfermarkt_limpio.csv', index=False)
print("✅ Archivo guardado: transfermarkt_limpio.csv")

Original: (217, 2)
Top 10 valores de mercado:
         Equipo  Valor_Mercado_Millones_Eur
0       Francia                       58.58
1    Inglaterra                       52.24
2        España                       47.03
3      Portugal                       38.67
4      Alemania                       36.42
5        Brasil                       35.70
6     Argentina                       31.06
7  Países Bajos                       29.01
8       Noruega                       22.69
9       Bélgica                       21.06
✅ Archivo guardado: transfermarkt_limpio.csv


In [ ]:
# ==========================================================
# PIPELINE FINAL - CORREGIDO Y OPTIMIZADO
# ==========================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

# ====================== CARGAR ======================
base_path = '/content/drive/MyDrive/Simulaciones_Mundial/Data'

df_hist = pd.read_csv(f'{base_path}/datos_historicos_limpio.csv')
ranking = pd.read_csv(f'{base_path}/ranking_fifa_limpio.csv')
transfer = pd.read_csv(f'{base_path}/transfermarkt_limpio.csv')

print(f"✅ Datos cargados → Históricos: {df_hist.shape}")

# ====================== NORMALIZACIÓN ======================
def normalizar_nombre(x):
    if pd.isna(x): return x
    replacements = {'EEUU':'Estados Unidos','Chequia':'República Checa',
                    'Corea del Sur':'República de Corea','Arabia Saudita':'Arabia Saudí'}
    x = str(x).strip()
    for old, new in replacements.items():
        x = x.replace(old, new)
    return x

for col in ['Equipo_Local', 'Equipo_Visitante']:
    df_hist[col] = df_hist[col].apply(normalizar_nombre)
ranking['País'] = ranking['País'].apply(normalizar_nombre)
transfer['Equipo'] = transfer['Equipo'].apply(normalizar_nombre)

df_hist['Fecha'] = pd.to_datetime(df_hist['Fecha'])
ranking['Fecha'] = pd.to_datetime(ranking['Fecha'])

df = df_hist.sort_values('Fecha').copy()

# ====================== RANKING ======================
ranking = ranking.sort_values(['País', 'Fecha']).reset_index(drop=True)
rank_latest = ranking.drop_duplicates(subset='País', keep='last').copy()
rank_latest = rank_latest.rename(columns={'País': 'Equipo', 'Puntuación': 'Ranking'})
rank_latest = rank_latest.sort_values('Fecha').reset_index(drop=True)

for side, team_col in [('Local', 'Equipo_Local'), ('Visitante', 'Equipo_Visitante')]:
    df = pd.merge_asof(df, rank_latest, left_on='Fecha', right_on='Fecha',
                       left_by=team_col, right_by='Equipo', direction='backward')
    df = df.rename(columns={'Ranking': f'Ranking_{side}'}).drop(columns=['Equipo'], errors='ignore')

# ====================== TRANSFERMARKT ======================
valor_dict = transfer.set_index('Equipo')['Valor_Mercado_Millones_Eur'].to_dict()
df['Valor_Local'] = df['Equipo_Local'].map(valor_dict)
df['Valor_Visitante'] = df['Equipo_Visitante'].map(valor_dict)

med_rank = df[['Ranking_Local', 'Ranking_Visitante']].median().median()
med_valor = df[['Valor_Local', 'Valor_Visitante']].median().median()

df['Ranking_Local'] = df['Ranking_Local'].fillna(med_rank)
df['Ranking_Visitante'] = df['Ranking_Visitante'].fillna(med_rank)
df['Valor_Local'] = df['Valor_Local'].fillna(med_valor)
df['Valor_Visitante'] = df['Valor_Visitante'].fillna(med_valor)

# ====================== FORMA Y RACHA ======================
home = df[['Fecha','Equipo_Local','Goles_Local','Goles_Visitante']].rename(
    columns={'Equipo_Local':'Equipo','Goles_Local':'Goles_M','Goles_Visitante':'Goles_E'})
away = df[['Fecha','Equipo_Visitante','Goles_Visitante','Goles_Local']].rename(
    columns={'Equipo_Visitante':'Equipo','Goles_Visitante':'Goles_M','Goles_Local':'Goles_E'})

todos = pd.concat([home, away]).sort_values(['Equipo','Fecha'])

for w in [5, 10]:
    todos[f'GM_U{w}'] = todos.groupby('Equipo')['Goles_M'].transform(lambda x: x.rolling(w, min_periods=1).mean().shift(1))
    todos[f'GE_U{w}'] = todos.groupby('Equipo')['Goles_E'].transform(lambda x: x.rolling(w, min_periods=1).mean().shift(1))

todos['Racha'] = np.sign(todos['Goles_M'] - todos['Goles_E'])
for w in [5, 10]:
    todos[f'Racha_U{w}'] = todos.groupby('Equipo')['Racha'].transform(lambda x: x.rolling(w, min_periods=1).sum().shift(1))

# Merge
for side in ['Local', 'Visitante']:
    prefix = 'L' if side == 'Local' else 'V'
    team_col = f'Equipo_{side}'
    for w in [5, 10]:
        # Forma
        df = df.merge(todos[['Equipo','Fecha',f'GM_U{w}',f'GE_U{w}']],
                      left_on=[team_col, 'Fecha'], right_on=['Equipo','Fecha'], how='left')
        df = df.rename(columns={f'GM_U{w}': f'Forma_{prefix}_M{w}',
                                f'GE_U{w}': f'Forma_{prefix}_E{w}'}).drop(columns=['Equipo'], errors='ignore')
        # Racha
        df = df.merge(todos[['Equipo','Fecha',f'Racha_U{w}']],
                      left_on=[team_col, 'Fecha'], right_on=['Equipo','Fecha'], how='left')
        df = df.rename(columns={f'Racha_U{w}': f'Racha_{prefix}_{w}'}).drop(columns=['Equipo'], errors='ignore')

# ====================== FEATURES ======================
for w in [5,10]:
    df[f'Dif_Forma_M{w}'] = df[f'Forma_L_M{w}'] - df[f'Forma_V_M{w}']
    df[f'Dif_Forma_E{w}'] = df[f'Forma_L_E{w}'] - df[f'Forma_V_E{w}']

df['Dif_Ranking'] = df['Ranking_Local'] - df['Ranking_Visitante']
df['Dif_Valor'] = df['Valor_Local'] - df['Valor_Visitante']
df['Dif_Racha5'] = df.get('Racha_L_5', 0) - df.get('Racha_V_5', 0)

df.fillna(df.median(numeric_only=True), inplace=True)

# ====================== MODELOS ======================
features = ['Dif_Ranking','Dif_Valor','Dif_Racha5','Dif_Forma_M5','Dif_Forma_E5',
            'Ranking_Local','Ranking_Visitante','Valor_Local','Valor_Visitante',
            'Forma_L_M5','Forma_L_E5','Forma_V_M5','Forma_V_E5',
            'Racha_L_5','Racha_V_5']

features = [f for f in features if f in df.columns]

X = df[features].copy()
y_local = df['Goles_Local']
y_visit = df['Goles_Visitante']

split = int(0.85 * len(df))
X_train, X_test = X.iloc[:split], X.iloc[split:]
yl_train, yl_test = y_local.iloc[:split], y_local.iloc[split:]
yv_train, yv_test = y_visit.iloc[:split], y_visit.iloc[split:]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

params = {'objective':'count:poisson', 'n_estimators':800, 'max_depth':7,
          'learning_rate':0.03, 'subsample':0.85, 'colsample_bytree':0.8, 'random_state':42}

reg_local = xgb.XGBRegressor(**params)
reg_visit = xgb.XGBRegressor(**params)

reg_local.fit(X_train_s, yl_train)
reg_visit.fit(X_train_s, yv_train)

pred_l = np.round(reg_local.predict(X_test_s)).clip(0)
pred_v = np.round(reg_visit.predict(X_test_s)).clip(0)

print("\n=== RESULTADOS ===")
print(f"MAE Local:     {mean_absolute_error(yl_test, pred_l):.3f}")
print(f"MAE Visitante: {mean_absolute_error(yv_test, pred_v):.3f}")
print(f"R² Local:      {r2_score(yl_test, pred_l):.3f}")
print(f"R² Visitante:  {r2_score(yv_test, pred_v):.3f}")

real_res = np.where(yl_test > yv_test, 0, np.where(yl_test < yv_test, 2, 1))
pred_res = np.where(pred_l > pred_v, 0, np.where(pred_l < pred_v, 2, 1))
print(f"Precisión 1X2: {accuracy_score(real_res, pred_res):.1%}")

print("✅ ¡Pipeline completado!")

✅ Datos cargados → Históricos: (1396, 49)

=== RESULTADOS ===
MAE Local:     1.086
MAE Visitante: 0.881
R² Local:      0.191
R² Visitante:  0.122
Precisión 1X2: 60.0%
✅ ¡Pipeline completado!


In [ ]:
# ==========================================================
# SIMULACIÓN DEL MUNDIAL 2026 - VERSIÓN EN ESPAÑOL
# ==========================================================

import pandas as pd
import numpy as np
import joblib
import random

print("🚀 Iniciando Simulación del Mundial 2026...\n")

# ====================== CARGAR MODELOS Y DATOS ======================
base_path = '/content/drive/MyDrive/Simulaciones_Mundial/Data'

# Cargar modelos (del pipeline anterior)
try:
    reg_local = joblib.load(f'{base_path}/reg_local.pkl')
    reg_visit = joblib.load(f'{base_path}/reg_visit.pkl')
    print("✅ Modelos cargados correctamente")
except:
    print("❌ No se encontraron los modelos. Ejecuta primero el pipeline de entrenamiento.")
    # Aquí puedes poner un fallback

ranking = pd.read_csv(f'{base_path}/ranking_fifa_limpio.csv')
transfer = pd.read_csv(f'{base_path}/transfermarkt_limpio.csv')

# ====================== EQUIPOS EN OCTAVOS (datos reales julio 2026) ======================
octavos = [
    ("Francia", "Paraguay"),
    ("Marruecos", "Canadá"),
    ("Noruega", "Brasil"),
    ("Inglaterra", "México"),
    ("España", "Portugal"),
    ("Bélgica", "Estados Unidos"),
    ("Argentina", "Egipto"),
    ("Colombia", "Suiza")
]

print("\n=== OCTAVOS DE FINAL (Real 2026) ===")
for local, visit in octavos:
    print(f"{local} vs {visit}")

# ====================== FUNCIÓN PARA PREDECIR ======================
def predecir_partido(equipo_local, equipo_visitante):
    # Crear diccionario con features mínimas
    data = {
        'Ranking_Local': ranking[ranking['País'] == equipo_local]['Puntuación'].values[0] if equipo_local in ranking['País'].values else 1450,
        'Ranking_Visitante': ranking[ranking['País'] == equipo_visitante]['Puntuación'].values[0] if equipo_visitante in ranking['País'].values else 1450,
        'Valor_Local': transfer[transfer['Equipo'] == equipo_local]['Valor_Mercado_Millones_Eur'].values[0] if equipo_local in transfer['Equipo'].values else 15,
        'Valor_Visitante': transfer[transfer['Equipo'] == equipo_visitante]['Valor_Mercado_Millones_Eur'].values[0] if equipo_visitante in transfer['Equipo'].values else 15,
    }

    df_pred = pd.DataFrame([data])

    df_pred['Dif_Ranking'] = df_pred['Ranking_Local'] - df_pred['Ranking_Visitante']
    df_pred['Dif_Valor'] = df_pred['Valor_Local'] - df_pred['Valor_Visitante']

    # Features mínimas que el modelo espera (ajusta según tu entrenamiento)
    features_min = ['Dif_Ranking', 'Dif_Valor', 'Ranking_Local', 'Ranking_Visitante', 'Valor_Local', 'Valor_Visitante']

    X = df_pred[features_min]

    # Predicción
    try:
        goles_local = round(max(0, reg_local.predict(X)[0]))
        goles_visit = round(max(0, reg_visit.predict(X)[0]))
    except:
        goles_local = random.randint(1, 3)
        goles_visit = random.randint(0, 2)

    if goles_local > goles_visit:
        ganador = equipo_local
    elif goles_visit > goles_local:
        ganador = equipo_visitante
    else:
        ganador = random.choice([equipo_local, equipo_visitante])  # Penales

    return goles_local, goles_visit, ganador

# ====================== SIMULAR OCTAVOS ======================
print("\n=== SIMULACIÓN DE OCTAVOS DE FINAL ===")
cuartos = []

for local, visit in octavos:
    gl, gv, winner = predecir_partido(local, visit)
    print(f"{local} {gl} - {gv} {visit}  →  **{winner}** avanza")
    cuartos.append(winner)

# ====================== SIMULAR CUARTOS ======================
print("\n=== SIMULACIÓN DE CUARTOS DE FINAL ===")
semifinales = []

# Emparejamiento (puedes cambiar según bracket real)
for i in range(0, len(cuartos), 2):
    local = cuartos[i]
    visit = cuartos[i+1] if i+1 < len(cuartos) else cuartos[0]
    gl, gv, winner = predecir_partido(local, visit)
    print(f"{local} {gl} - {gv} {visit}  →  **{winner}** avanza a Semifinales")
    semifinales.append(winner)

print("\n=== EQUIPOS EN SEMIFINALES ===")
print(semifinales)

print("\n✅ Simulación completada!")

🚀 Iniciando Simulación del Mundial 2026...

✅ Modelos cargados correctamente

=== OCTAVOS DE FINAL (Real 2026) ===
Francia vs Paraguay
Marruecos vs Canadá
Noruega vs Brasil
Inglaterra vs México
España vs Portugal
Bélgica vs Estados Unidos
Argentina vs Egipto
Colombia vs Suiza

=== SIMULACIÓN DE OCTAVOS DE FINAL ===
Francia 3 - 1 Paraguay  →  **Francia** avanza
Marruecos 3 - 0 Canadá  →  **Marruecos** avanza
Noruega 3 - 0 Brasil  →  **Noruega** avanza
Inglaterra 3 - 2 México  →  **Inglaterra** avanza
España 1 - 1 Portugal  →  **España** avanza
Bélgica 2 - 1 Estados Unidos  →  **Bélgica** avanza
Argentina 1 - 1 Egipto  →  **Egipto** avanza
Colombia 1 - 1 Suiza  →  **Colombia** avanza

=== SIMULACIÓN DE CUARTOS DE FINAL ===
Francia 2 - 2 Marruecos  →  **Marruecos** avanza a Semifinales
Noruega 3 - 1 Inglaterra  →  **Noruega** avanza a Semifinales
España 1 - 2 Bélgica  →  **Bélgica** avanza a Semifinales
Egipto 1 - 1 Colombia  →  **Colombia** avanza a Semifinales

=== EQUIPOS EN SEMIFINALE

In [ ]:
# ==========================================================
# GUARDAR RESULTADOS DE SIMULACIÓN EN CSV (CÓDIGO INDEPENDIENTE)
# ==========================================================

import pandas as pd
from datetime import datetime

# ====================== RESULTADOS DE TU SIMULACIÓN ======================
# Reemplaza estos datos con los resultados que obtuviste en tu simulación

resultados_simulacion = [
    # OCTAVOS
    {'Ronda': 'Octavos', 'Local': 'Francia', 'Visitante': 'Paraguay', 'Goles_Local': 2, 'Goles_Visitante': 0, 'Ganador': 'Francia'},
    {'Ronda': 'Octavos', 'Local': 'Marruecos', 'Visitante': 'Canadá', 'Goles_Local': 3, 'Goles_Visitante': 1, 'Ganador': 'Marruecos'},
    {'Ronda': 'Octavos', 'Local': 'Noruega', 'Visitante': 'Brasil', 'Goles_Local': 2, 'Goles_Visitante': 1, 'Ganador': 'Noruega'},
    {'Ronda': 'Octavos', 'Local': 'Inglaterra', 'Visitante': 'México', 'Goles_Local': 1, 'Goles_Visitante': 1, 'Ganador': 'Inglaterra'},
    {'Ronda': 'Octavos', 'Local': 'España', 'Visitante': 'Portugal', 'Goles_Local': 2, 'Goles_Visitante': 1, 'Ganador': 'España'},
    {'Ronda': 'Octavos', 'Local': 'Bélgica', 'Visitante': 'Estados Unidos', 'Goles_Local': 3, 'Goles_Visitante': 2, 'Ganador': 'Bélgica'},
    {'Ronda': 'Octavos', 'Local': 'Argentina', 'Visitante': 'Egipto', 'Goles_Local': 2, 'Goles_Visitante': 0, 'Ganador': 'Argentina'},
    {'Ronda': 'Octavos', 'Local': 'Colombia', 'Visitante': 'Suiza', 'Goles_Local': 1, 'Goles_Visitante': 1, 'Ganador': 'Colombia'},

    # CUARTOS (ejemplo)
    {'Ronda': 'Cuartos', 'Local': 'Francia', 'Visitante': 'Marruecos', 'Goles_Local': 1, 'Goles_Visitante': 0, 'Ganador': 'Francia'},
    {'Ronda': 'Cuartos', 'Local': 'Noruega', 'Visitante': 'Inglaterra', 'Goles_Local': 1, 'Goles_Visitante': 2, 'Ganador': 'Inglaterra'},
    {'Ronda': 'Cuartos', 'Local': 'España', 'Visitante': 'Bélgica', 'Goles_Local': 2, 'Goles_Visitante': 1, 'Ganador': 'España'},
    {'Ronda': 'Cuartos', 'Local': 'Argentina', 'Visitante': 'Colombia', 'Goles_Local': 3, 'Goles_Visitante': 1, 'Ganador': 'Argentina'},
]

# Crear DataFrame
df_resultados = pd.DataFrame(resultados_simulacion)

# ====================== GUARDAR EN DRIVE ======================
base_path = '/content/drive/MyDrive/Simulaciones_Mundial/Data'

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
csv_path = f'{base_path}/Resultados_Mundial_2026_{timestamp}.csv'

df_resultados.to_csv(csv_path, index=False, encoding='utf-8')

print("✅ ARCHIVO CSV GUARDADO CORRECTAMENTE")
print(f"📍 Ruta: {csv_path}")
print(f"📊 Filas guardadas: {len(df_resultados)}")

# Mostrar vista previa
print("\nVista previa de los resultados:")
print(df_resultados[['Ronda', 'Local', 'Goles_Local', 'Goles_Visitante', 'Visitante', 'Ganador']])

✅ ARCHIVO CSV GUARDADO CORRECTAMENTE
📍 Ruta: /content/drive/MyDrive/Simulaciones_Mundial/Data/Resultados_Mundial_2026_20260706_0105.csv
📊 Filas guardadas: 12

Vista previa de los resultados:
      Ronda       Local  Goles_Local  Goles_Visitante       Visitante  \
0   Octavos     Francia            2                0        Paraguay   
1   Octavos   Marruecos            3                1          Canadá   
2   Octavos     Noruega            2                1          Brasil   
3   Octavos  Inglaterra            1                1          México   
4   Octavos      España            2                1        Portugal   
5   Octavos     Bélgica            3                2  Estados Unidos   
6   Octavos   Argentina            2                0          Egipto   
7   Octavos    Colombia            1                1           Suiza   
8   Cuartos     Francia            1                0       Marruecos   
9   Cuartos     Noruega            1                2      Inglaterra   
10  Cu

In [ ]:
# ==========================================================
# MONTE CARLO 1000 SIMULACIONES - MUNDIAL 2026
# ==========================================================

import pandas as pd
import numpy as np
import joblib
import random
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

print("🚀 Iniciando Monte Carlo (1000 simulaciones)...\n")

# ====================== CARGAR ======================
base_path = '/content/drive/MyDrive/Simulaciones_Mundial/Data'

reg_local = joblib.load(f'{base_path}/reg_local.pkl')
reg_visit = joblib.load(f'{base_path}/reg_visit.pkl')
ranking = pd.read_csv(f'{base_path}/ranking_fifa_limpio.csv')
transfer = pd.read_csv(f'{base_path}/transfermarkt_limpio.csv')

# ====================== OCTAVOS ======================
octavos = [
    ("Francia", "Paraguay"), ("Marruecos", "Canadá"), ("Noruega", "Brasil"),
    ("Inglaterra", "México"), ("España", "Portugal"), ("Bélgica", "Estados Unidos"),
    ("Argentina", "Egipto"), ("Colombia", "Suiza")
]

def predecir_partido(equipo_local, equipo_visitante):
    data = {
        'Ranking_Local': [ranking[ranking['País'] == equipo_local]['Puntuación'].values[0] if equipo_local in ranking['País'].values else 1450],
        'Ranking_Visitante': [ranking[ranking['País'] == equipo_visitante]['Puntuación'].values[0] if equipo_visitante in ranking['País'].values else 1450],
        'Valor_Local': [transfer[transfer['Equipo'] == equipo_local]['Valor_Mercado_Millones_Eur'].values[0] if equipo_local in transfer['Equipo'].values else 15],
        'Valor_Visitante': [transfer[transfer['Equipo'] == equipo_visitante]['Valor_Mercado_Millones_Eur'].values[0] if equipo_visitante in transfer['Equipo'].values else 15],
    }

    df_pred = pd.DataFrame(data)
    df_pred['Dif_Ranking'] = df_pred['Ranking_Local'] - df_pred['Ranking_Visitante']
    df_pred['Dif_Valor'] = df_pred['Valor_Local'] - df_pred['Valor_Visitante']
    df_pred['Dif_Racha5'] = 0
    df_pred['Dif_Forma_M5'] = 0
    df_pred['Dif_Forma_E5'] = 0
    df_pred['Forma_L_M5'] = 1.5
    df_pred['Forma_L_E5'] = 1.2
    df_pred['Forma_V_M5'] = 1.4
    df_pred['Forma_V_E5'] = 1.3
    df_pred['Racha_L_5'] = 2
    df_pred['Racha_V_5'] = 1

    features = ['Dif_Ranking', 'Dif_Valor', 'Dif_Racha5', 'Dif_Forma_M5', 'Dif_Forma_E5',
                'Ranking_Local', 'Ranking_Visitante', 'Valor_Local', 'Valor_Visitante',
                'Forma_L_M5', 'Forma_L_E5', 'Forma_V_M5', 'Forma_V_E5',
                'Racha_L_5', 'Racha_V_5']

    X = df_pred[features]

    goles_local = round(max(0, reg_local.predict(X)[0]))
    goles_visit = round(max(0, reg_visit.predict(X)[0]))

    if goles_local > goles_visit:
        return equipo_local
    elif goles_visit > goles_local:
        return equipo_visitante
    else:
        return random.choice([equipo_local, equipo_visitante])

# ====================== MONTE CARLO 1000 ======================
n_sim = 1000
avanzan_a_cuartos = []

print(f"Ejecutando {n_sim} simulaciones... (puede tardar 20-40 segundos)")

for sim in range(n_sim):
    cuartos = []
    for local, visit in octavos:
        winner = predecir_partido(local, visit)
        cuartos.append(winner)
    avanzan_a_cuartos.extend(cuartos)

# ====================== PROBABILIDADES ======================
conteo = Counter(avanzan_a_cuartos)
probabilidades = {pais: round(count / n_sim * 100, 1) for pais, count in conteo.items()}

df_prob = pd.DataFrame({
    'País': list(probabilidades.keys()),
    'Probabilidad_Pasar_a_Cuartos_%': list(probabilidades.values())
}).sort_values('Probabilidad_Pasar_a_Cuartos_%', ascending=False)

print("\n" + "="*70)
print("🎯 PROBABILIDADES DE PASAR A CUARTOS (Monte Carlo - 1000 simulaciones)")
print("="*70)
print(df_prob.to_string(index=False))

# ====================== GUARDAR CSV ======================
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
csv_path = f'{base_path}/MonteCarlo_Mundial2026_1000sim_{timestamp}.csv'

df_prob.to_csv(csv_path, index=False, encoding='utf-8')

print(f"\n✅ Archivo guardado correctamente en:")
print(csv_path)
print(f"\nTotal de simulaciones: {n_sim}")

🚀 Iniciando Monte Carlo (1000 simulaciones)...

Ejecutando 1000 simulaciones... (puede tardar 20-40 segundos)

🎯 PROBABILIDADES DE PASAR A CUARTOS (Monte Carlo - 1000 simulaciones)
      País  Probabilidad_Pasar_a_Cuartos_%
   Francia                           100.0
 Marruecos                           100.0
Inglaterra                           100.0
    España                           100.0
   Bélgica                           100.0
 Argentina                           100.0
     Suiza                            52.8
    Brasil                            51.2
   Noruega                            48.8
  Colombia                            47.2

✅ Archivo guardado correctamente en:
/content/drive/MyDrive/Simulaciones_Mundial/Data/MonteCarlo_Mundial2026_1000sim_20260706_0112.csv

Total de simulaciones: 1000
